# 01: Data Collection

This notebook collects all raw data needed for our NBA player injury prediction project.

**Data Sources:**
1. **elap733/NBA-Injuries-Analysis** — Pre-scraped injury data from prosportstransactions.com (2010–2019)
2. **nba_api** — Official NBA statistics API for player stats, bio data, tracking metrics, and team schedules

**Output:** Raw CSV files saved to `data/raw/elap733/` and `data/raw/nba_api/`

In [1]:
import os
import time
import pandas as pd
import numpy as np
import urllib.request
from pathlib import Path

# === Configuration ===
FIRST_SEASON = 2013        # First season to collect (2013-14)
LAST_SEASON = 2018         # Last season to collect (2018-19)

RAW_ELAP_DIR = '../data/raw/elap733'
RAW_NBA_DIR = '../data/raw/nba_api'

# Create output directories
Path(RAW_ELAP_DIR).mkdir(parents=True, exist_ok=True)
Path(RAW_NBA_DIR).mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 50)
print(f"Collecting seasons: {FIRST_SEASON}-{FIRST_SEASON+1} through {LAST_SEASON}-{LAST_SEASON+1}")

What this does: Sets up everything we need. The key constants define our season range: training on 2013–2017 seasons and test on 2018-19. We also  create the folder structure for raw data, where we'll put the selected seasons from both data sources.

---
## Section 1: Download elap733 Injury Data

The [elap733/NBA-Injuries-Analysis](https://github.com/elap733/NBA-Injuries-Analysis) repo contains pre-scraped injury data from prosportstransactions.com. 

We'll download 4 CSV files, which will contain the seasons 2010-2019 of injury data. Not all data will be utilized of course, and the key one is **missed_games** which becomes our **target variable** (games missed per player due to injury).

In [2]:
ELAP_BASE_URL = "https://raw.githubusercontent.com/elap733/NBA-Injuries-Analysis/master/data/01_raw"

ELAP_FILES = {
    "missed_games_2010_2019.csv": f"{ELAP_BASE_URL}/prosportstransactions_scrape_missedgames_2010_2019.csv",
    "injury_transactions_2010_2019.csv": f"{ELAP_BASE_URL}/prosportstransactions_scrape_IRL_2010_2019.csv",
    "player_stats_1994_2019.csv": f"{ELAP_BASE_URL}/player_stats_1994_2019.csv",
    "team_schedules_2010_2020.csv": f"{ELAP_BASE_URL}/all_teams_schedule_2010_2020.csv",
}

for local_name, url in ELAP_FILES.items():
    filepath = f"{RAW_ELAP_DIR}/{local_name}"
    if os.path.exists(filepath):
        print(f"[SKIP] {local_name} already exists")
    else:
        print(f"Downloading {local_name}...")
        urllib.request.urlretrieve(url, filepath)
        print(f"  Saved to {filepath}")

print("\nAll elap733 files downloaded.")

[SKIP] missed_games_2010_2019.csv already exists
[SKIP] injury_transactions_2010_2019.csv already exists
[SKIP] player_stats_1994_2019.csv already exists
[SKIP] team_schedules_2010_2020.csv already exists

All elap733 files downloaded.


In [3]:
# Checking if files were created: Load and inspect each downloaded file
for filename in ELAP_FILES.keys():
    df = pd.read_csv(f"{RAW_ELAP_DIR}/{filename}", index_col=0)
    print(f"{filename}: {df.shape[0]} rows, {df.shape[1]} cols")
    print(f"  Columns: {list(df.columns)}")
    print()

missed_games_2010_2019.csv: 11531 rows, 5 cols
  Columns: ['Date', 'Team', 'Acquired', 'Relinquished', 'Notes']

injury_transactions_2010_2019.csv: 14135 rows, 5 cols
  Columns: ['Date', 'Team', 'Acquired', 'Relinquished', 'Notes']

player_stats_1994_2019.csv: 19786 rows, 31 cols
  Columns: ['Year', 'Season', 'Player', 'Pos', 'Age', 'Tm', 'G', 'GS', 'MP', 'FG', 'FGA', 'FG%', '3P', '3PA', '3P%', '2P', '2PA', '2P%', 'eFG%', 'FT', 'FTA', 'FT%', 'ORB', 'DRB', 'TRB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PTS']

team_schedules_2010_2020.csv: 28240 rows, 8 cols
  Columns: ['Team', 'Year', 'Season', 'Game_num', 'Date', 'Away_flag', 'Opponent', 'OT_flag']



Now with our elap77 injury data, here's a quick recap of what we have:

- missed_games: 11,531 rows — our target variable source

- injury_transactions: 14,135 rows — reference data

- player_stats: 19,786 rows — historical stats (we'll use nba_api instead for richer data)

- team_schedules: 28,240 rows — for back-to-back calculations

---
## Section 2: Pull Player Season Stats from nba_api

We'll use the `nba_api` package to get per-game player stats (points, rebounds, assists, minutes, usage rate, etc.) for each season. These are our primary features for our models. We pull one CSV per season.

**Endpoint for API:** `LeagueDashPlayerStats` — returns per-game averages for all players in a given season.

For each season from 2013-14 through 2018-19 **(our targeted season range)**, the cell above calls the NBA stats API and saves per-game averages (minutes, points, rebounds, etc.) for every player

In [4]:
from nba_api.stats.endpoints import leaguedashplayerstats

print("Pulling player season stats from nba_api...")

for season in range(FIRST_SEASON, LAST_SEASON + 1):
    season_str = f"{season}-{str(season+1)[-2:]}"
    filepath = f"{RAW_NBA_DIR}/player_stats_{season}.csv"
    
    if os.path.exists(filepath):
        print(f"[SKIP] {season_str} already exists")
        continue
    
    print(f"Fetching {season_str}...", end=" ")
    result = leaguedashplayerstats.LeagueDashPlayerStats(
        season=season_str, per_mode_detailed='PerGame'
    )
    df = result.get_data_frames()[0]
    df['SEASON'] = season_str
    df.to_csv(filepath, index=False)
    print(f"{len(df)} players")
    time.sleep(1)  # Rate limiting — be nice to the API

print("\nDone.")

Pulling player season stats from nba_api...
[SKIP] 2013-14 already exists
[SKIP] 2014-15 already exists
[SKIP] 2015-16 already exists
[SKIP] 2016-17 already exists
[SKIP] 2017-18 already exists
[SKIP] 2018-19 already exists

Done.


---
## Section 3: Pull Player Bio Data from nba_api

Now that we have our season stats, we'll pull player demographics (height, weight, age, draft info, years of experience) plus advanced rate stats (USG%, TS%, etc.). These will serve as both features and control variables.

In [5]:
from nba_api.stats.endpoints import leaguedashplayerbiostats

print("Pulling player bio stats from nba_api...")

for season in range(FIRST_SEASON, LAST_SEASON + 1):
    season_str = f"{season}-{str(season+1)[-2:]}"
    filepath = f"{RAW_NBA_DIR}/player_bio_{season}.csv"
    
    if os.path.exists(filepath):
        print(f"[SKIP] {season_str} already exists")
        continue
    
    print(f"Fetching {season_str}...", end=" ")
    result = leaguedashplayerbiostats.LeagueDashPlayerBioStats(
        season=season_str, per_mode_simple='PerGame'
    )
    df = result.get_data_frames()[0]
    df['SEASON'] = season_str
    df.to_csv(filepath, index=False)
    print(f"{len(df)} players")
    time.sleep(1)

print("\nDone.")

Pulling player bio stats from nba_api...
[SKIP] 2013-14 already exists
[SKIP] 2014-15 already exists
[SKIP] 2015-16 already exists
[SKIP] 2016-17 already exists
[SKIP] 2017-18 already exists
[SKIP] 2018-19 already exists

Done.


---
## Section 4: Pull Player Tracking Stats from nba_api

Let's pull physical workload metrics (speed, distance traveled per game) that traditional box scores miss. Available from 2013-14 onward, which is exactly our range, so no missing data.

These stats are niche and could fit well as our features.

In [6]:
from nba_api.stats.endpoints import leaguedashptstats

print("Pulling player tracking stats from nba_api...")

for season in range(FIRST_SEASON, LAST_SEASON + 1):
    season_str = f"{season}-{str(season+1)[-2:]}"
    filepath = f"{RAW_NBA_DIR}/tracking_stats_{season}.csv"
    
    if os.path.exists(filepath):
        print(f"[SKIP] {season_str} already exists")
        continue
    
    print(f"Fetching {season_str}...", end=" ")
    result = leaguedashptstats.LeagueDashPtStats(
        season=season_str, per_mode_simple='PerGame', player_or_team='Player'
    )
    df = result.get_data_frames()[0]
    df['SEASON'] = season_str
    df.to_csv(filepath, index=False)
    print(f"{len(df)} players")
    time.sleep(1)

print("\nDone.")

Pulling player tracking stats from nba_api...
[SKIP] 2013-14 already exists
[SKIP] 2014-15 already exists
[SKIP] 2015-16 already exists
[SKIP] 2016-17 already exists
[SKIP] 2017-18 already exists
[SKIP] 2018-19 already exists

Done.


---
## Section 5: Pull Team Schedules from nba_api

Now finally, we'll pull game schedules that will let us calculate back-to-back games (a well-established injury risk factor.) We pull every team's game log for each season.

In [7]:
from nba_api.stats.endpoints import teamgamelog
from nba_api.stats.static import teams

nba_teams = teams.get_teams()
print(f"Found {len(nba_teams)} NBA teams")
print("Pulling team schedules from nba_api...")

for season in range(FIRST_SEASON, LAST_SEASON + 1):
    season_str = f"{season}-{str(season+1)[-2:]}"
    filepath = f"{RAW_NBA_DIR}/team_schedules_{season}.csv"
    
    if os.path.exists(filepath):
        print(f"[SKIP] {season_str} already exists")
        continue
    
    print(f"Fetching {season_str}...", end=" ")
    season_dfs = []
    for team in nba_teams:
        result = teamgamelog.TeamGameLog(
            team_id=team['id'], season=season_str
        )
        df = result.get_data_frames()[0]
        df['TEAM_NAME'] = team['full_name']
        season_dfs.append(df)
        time.sleep(0.5)
    
    df_season = pd.concat(season_dfs, ignore_index=True)
    df_season['SEASON'] = season_str
    df_season.to_csv(filepath, index=False)
    print(f"{len(df_season)} games")

print("\nDone.")

Found 30 NBA teams
Pulling team schedules from nba_api...
Fetching 2013-14... 2460 games
Fetching 2014-15... 2460 games
Fetching 2015-16... 2460 games
Fetching 2016-17... 2460 games
Fetching 2017-18... 2460 games
Fetching 2018-19... 2460 games

Done.


---
## Validation & Summary of Notebook

Verify all expected files exist and have reasonable row counts.

In [8]:
print("=" * 60)
print("DATA COLLECTION SUMMARY")
print("=" * 60)

# Check elap733 files
print("\nelap733 files:")
for filename in ELAP_FILES.keys():
    df = pd.read_csv(f"{RAW_ELAP_DIR}/{filename}", index_col=0)
    print(f"  {filename}: {df.shape[0]:,} rows, {df.shape[1]} cols")

# Check nba_api files
print("\nnba_api files:")
for prefix in ['player_stats', 'player_bio', 'tracking_stats', 'team_schedules']:
    for season in range(FIRST_SEASON, LAST_SEASON + 1):
        filepath = f"{RAW_NBA_DIR}/{prefix}_{season}.csv"
        if os.path.exists(filepath):
            df = pd.read_csv(filepath)
            print(f"  {prefix}_{season}.csv: {df.shape[0]:,} rows")
        else:
            print(f"  [MISSING] {prefix}_{season}.csv")

print("\nNB01 complete. All raw data ready for cleaning.")

DATA COLLECTION SUMMARY

elap733 files:
  missed_games_2010_2019.csv: 11,531 rows, 5 cols
  injury_transactions_2010_2019.csv: 14,135 rows, 5 cols
  player_stats_1994_2019.csv: 19,786 rows, 31 cols
  team_schedules_2010_2020.csv: 28,240 rows, 8 cols

nba_api files:
  player_stats_2013.csv: 482 rows
  player_stats_2014.csv: 492 rows
  player_stats_2015.csv: 476 rows
  player_stats_2016.csv: 486 rows
  player_stats_2017.csv: 540 rows
  player_stats_2018.csv: 530 rows
  player_bio_2013.csv: 482 rows
  player_bio_2014.csv: 492 rows
  player_bio_2015.csv: 476 rows
  player_bio_2016.csv: 486 rows
  player_bio_2017.csv: 540 rows
  player_bio_2018.csv: 530 rows
  tracking_stats_2013.csv: 482 rows
  tracking_stats_2014.csv: 492 rows
  tracking_stats_2015.csv: 476 rows
  tracking_stats_2016.csv: 486 rows
  tracking_stats_2017.csv: 540 rows
  tracking_stats_2018.csv: 530 rows
  team_schedules_2013.csv: 2,460 rows
  team_schedules_2014.csv: 2,460 rows
  team_schedules_2015.csv: 2,460 rows
  team_s

---
## Summary & Handoff to NB02

### What we did in this notebook

- Downloaded 4 CSV files from the **elap733/NBA-Injuries-Analysis** GitHub repo (injury transactions, missed games, player stats, team schedules covering 2010–2019)

- Pulled **player season stats** from nba_api (per-game averages for all players, 2013–2018)

- Pulled **player bio data** from nba_api (height, weight, draft info, advanced rate stats)

- Pulled **tracking stats** from nba_api (speed, distance traveled per game)

- Pulled **team schedules** from nba_api (game logs for all 30 teams, 2013–2018)

- All raw files saved to `data/raw/elap733/` and `data/raw/nba_api/`

### Next: NB02 (Data Cleaning)

1. Clean injury data → build target variable (games missed per player-season)

2. Combine and clean nba_api files across seasons

3. Map player names between elap733 and nba_api datasets

4. Calculate back-to-back games from schedules